# 📊 Exploratory Data Analysis (EDA) using Python

**Project:** TechX Sales Dataset  
**Author:** Manas Aswal

This notebook performs data inspection, cleaning checks, descriptive statistics, visualization, correlation analysis, outlier detection, and business-oriented trend analysis.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 1. Load the Dataset

Place `TechX_Sales_Dataset.csv` in the same folder as this notebook.


In [ ]:
DATA_FILE = "TechX_Sales_Dataset.csv"

df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


## 2. Dataset Structure and Data Types


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isna().sum().values,
    "Unique Values": [df[c].nunique(dropna=True) for c in df.columns]
}))


In [ ]:
df.info()


## 3. Descriptive Statistics


In [ ]:
display(df.describe(include="all").T)


## 4. Missing Values


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percentage": missing_pct.round(2)
})

display(missing_report[missing_report["Missing Count"] > 0])

if missing.sum() == 0:
    print("No missing values were found.")


## 5. Duplicate Records


In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate records:", duplicate_count)

if duplicate_count > 0:
    display(df[df.duplicated(keep=False)].head())


## 6. Data Cleaning and Preparation

The following steps standardize column names and attempt to convert common date and numeric fields. The original dataset is retained in memory as `df`.


In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)

# Remove exact duplicate records if present.
df = df.drop_duplicates().copy()

# Convert common date columns when available.
date_candidates = [c for c in df.columns if "date" in c]
for col in date_candidates:
    converted = pd.to_datetime(df[col], errors="coerce")
    if converted.notna().sum() > 0:
        df[col] = converted

# Convert common numeric columns when available.
numeric_candidates = [
    c for c in df.columns
    if any(k in c for k in ["sales", "profit", "quantity", "amount", "revenue"])
]
for col in numeric_candidates:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False).str.replace("₹", "", regex=False),
        errors="coerce"
    )

print("Cleaned shape:", df.shape)
print("Columns:", list(df.columns))


## 7. Numerical and Categorical Features


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


## 8. Unique Values in Categorical Columns


In [ ]:
for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", df[col].nunique(dropna=True))
    print(df[col].value_counts(dropna=False).head(10))


## 9. Sales Distribution

If a sales/revenue column is available, the notebook plots its distribution and summarizes it statistically.


In [ ]:
sales_col = next((c for c in df.columns if c in ["sales", "revenue", "sales_amount", "amount"]), None)

if sales_col:
    print(df[sales_col].describe())
    plt.figure(figsize=(10, 5))
    sns.histplot(df[sales_col].dropna(), kde=True)
    plt.title("Sales Distribution")
    plt.xlabel(sales_col.title())
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
else:
    print("No standard sales/revenue column was detected.")


## 10. Sales Boxplot and Outlier Inspection


In [ ]:
if sales_col:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[sales_col].dropna())
    plt.title("Sales Boxplot")
    plt.xlabel(sales_col.title())
    plt.tight_layout()
    plt.show()

    q1 = df[sales_col].quantile(0.25)
    q3 = df[sales_col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[(df[sales_col] < lower) | (df[sales_col] > upper)]
    print("Potential sales outliers:", len(outliers))
else:
    print("Sales column not detected.")


## 11. Sales vs Profit Relationship


In [ ]:
profit_col = next((c for c in df.columns if c in ["profit", "profit_amount"]), None)

if sales_col and profit_col:
    plt.figure(figsize=(9, 6))
    sns.scatterplot(data=df, x=sales_col, y=profit_col)
    plt.title("Sales vs Profit")
    plt.xlabel(sales_col.title())
    plt.ylabel(profit_col.title())
    plt.tight_layout()
    plt.show()

    print("Correlation between sales and profit:",
          round(df[[sales_col, profit_col]].corr().iloc[0, 1], 3))
else:
    print("Sales and/or profit column not detected.")


## 12. Correlation Heatmap


In [ ]:
if len(numeric_cols) >= 2:
    plt.figure(figsize=(10, 7))
    corr = df[numeric_cols].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numerical columns are required.")


## 13. Pair Plot

A pair plot is generated only when the dataset has a manageable number of numerical variables.


In [ ]:
pair_cols = numeric_cols[:5]

if len(pair_cols) >= 2:
    sns.pairplot(df[pair_cols].dropna().sample(
        min(1000, len(df[pair_cols].dropna())), random_state=42
    ))
    plt.show()
else:
    print("Not enough numerical columns for a pair plot.")


## 14. Sales by Region


In [ ]:
region_col = next((c for c in df.columns if c == "region" or "region" in c), None)

if region_col and sales_col:
    region_sales = df.groupby(region_col)[sales_col].sum().sort_values(ascending=False)
    display(region_sales.to_frame("Total Sales"))

    plt.figure(figsize=(10, 5))
    sns.barplot(x=region_sales.index, y=region_sales.values)
    plt.title("Sales by Region")
    plt.xlabel("Region")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("Region and/or sales column not detected.")


## 15. Sales by Category


In [ ]:
category_col = next((c for c in df.columns if c == "category" or "category" in c), None)

if category_col and sales_col:
    category_sales = df.groupby(category_col)[sales_col].sum().sort_values(ascending=False)
    display(category_sales.to_frame("Total Sales"))

    plt.figure(figsize=(10, 5))
    sns.barplot(x=category_sales.index, y=category_sales.values)
    plt.title("Sales by Category")
    plt.xlabel("Category")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("Category and/or sales column not detected.")


## 16. Monthly Sales Trend


In [ ]:
order_date_col = next((c for c in df.columns if c == "order_date" or "date" in c), None)

if order_date_col and sales_col:
    temp = df.dropna(subset=[order_date_col]).copy()
    temp["month"] = temp[order_date_col].dt.to_period("M").astype(str)
    monthly_sales = temp.groupby("month")[sales_col].sum()

    display(monthly_sales.to_frame("Total Sales"))

    plt.figure(figsize=(12, 5))
    sns.lineplot(x=monthly_sales.index, y=monthly_sales.values, marker="o")
    plt.title("Monthly Sales Trend")
    plt.xlabel("Month")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Order date and/or sales column not detected.")


## 17. Monthly Profit Trend


In [ ]:
if order_date_col and profit_col:
    temp = df.dropna(subset=[order_date_col]).copy()
    temp["month"] = temp[order_date_col].dt.to_period("M").astype(str)
    monthly_profit = temp.groupby("month")[profit_col].sum()

    display(monthly_profit.to_frame("Total Profit"))

    plt.figure(figsize=(12, 5))
    sns.lineplot(x=monthly_profit.index, y=monthly_profit.values, marker="o")
    plt.title("Monthly Profit Trend")
    plt.xlabel("Month")
    plt.ylabel("Total Profit")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Order date and/or profit column not detected.")


## 18. Top Products by Sales


In [ ]:
product_col = next((c for c in df.columns if c == "product" or "product" in c), None)

if product_col and sales_col:
    top_products = (
        df.groupby(product_col)[sales_col]
          .sum()
          .sort_values(ascending=False)
          .head(10)
    )
    display(top_products.to_frame("Total Sales"))

    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_products.values, y=top_products.index)
    plt.title("Top 10 Products by Sales")
    plt.xlabel("Total Sales")
    plt.ylabel("Product")
    plt.tight_layout()
    plt.show()
else:
    print("Product and/or sales column not detected.")


## 19. Business Summary

The final conclusions should be based on the actual notebook outputs. Typical questions to answer are:

1. Which region generates the highest sales?
2. Which category contributes the most sales?
3. Which products generate the highest sales?
4. Is there a positive relationship between sales and profit?
5. Which months show unusually high or low sales?
6. Are there significant outliers in sales or profit?
7. Which variables have the strongest numerical correlations?


In [ ]:
print("EDA completed successfully.")
print("Final dataset shape:", df.shape)

if sales_col:
    print("Total Sales:", round(df[sales_col].sum(), 2))
if profit_col:
    print("Total Profit:", round(df[profit_col].sum(), 2))


## 20. Conclusion

This EDA provides a structured view of the TechX Sales Dataset by combining data-quality checks, descriptive statistics, visualization, correlation analysis, outlier detection, and business-oriented trend analysis.

The outputs from the notebook can be used to support data-driven decisions around regional performance, product performance, category contribution, profitability, and time-based sales patterns.
